In [6]:
import os
from dotenv import load_dotenv
from langchain_cerebras import ChatCerebras
from langchain_groq import ChatGroq
import resend
load_dotenv()
os.environ["CEREBRAS_API_KEY"]=os.getenv("CEREBRAS_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
resend.api_key=os.getenv("RESEND_API_KEY")


llm_gpt= ChatCerebras( model="gpt-oss-120b",
    api_key=os.getenv("CEREBRAS_API_KEY"))

llm_groq=ChatGroq(model="llama-3.1-8b-instant",
api_key=os.getenv("GROQ_API_KEY"))

#response_gpt =llm_gpt.invoke("Hi mate!")
#response_groq =llm_groq.invoke("Hi mate!")
#print(response_gpt.content)
#print(response_groq.content)

In [ ]:
import os
import resend

resend.api_key = os.getenv("RESEND_API_KEY")

params: resend.Emails.SendParams = {
  "from": "Acme <onboarding@resend.dev>",
  "to": ["lokeshvijayraina@gmail.com"], 
  "subject": "hello world",
  "html": "<p>it works!</p>"
}

email = resend.Emails.send(params)
print(email)


{'id': '416daac3-97df-4208-b192-476dede86d39', 'http_headers': {'Date': 'Sun, 05 Jul 2026 07:17:51 GMT', 'Content-Type': 'application/json', 'Content-Length': '45', 'Connection': 'keep-alive', 'Ratelimit-Limit': '10', 'Ratelimit-Policy': '10;w=1', 'Ratelimit-Remaining': '9', 'Ratelimit-Reset': '1', 'X-Resend-Daily-Quota': '3', 'X-Resend-Monthly-Quota': '3', 'Cf-Cache-Status': 'DYNAMIC', 'Server': 'cloudflare', 'CF-RAY': 'a16497a3db271796-MAA'}}


In [24]:
import os
import sib_api_v3_sdk
from sib_api_v3_sdk.rest import ApiException
from dotenv import load_dotenv

load_dotenv()

BREVO_KEY = os.getenv("BREVO_API_KEY")

if not BREVO_KEY:
    raise ValueError("Error: BREVO_API_KEY is missing!")

configuration = sib_api_v3_sdk.Configuration()
configuration.api_key['api-key'] = BREVO_KEY

api_instance = sib_api_v3_sdk.TransactionalEmailsApi(sib_api_v3_sdk.ApiClient(configuration))

send_smtp_email = sib_api_v3_sdk.SendSmtpEmail(
    sender={"name": "Assistant", "email": "lokeshvijayraina@gmail.com"},
    to=[{"email": "itslokeshx@gmail.com", "name": "Recipient"}],
    subject="LookOut here1",
    html_content="<h3>Success!</h3><p>The API key is working.</p>"
)

try:
    api_response = api_instance.send_transac_email(send_smtp_email)
    print("Success! Message ID:", api_response.message_id)
except ApiException as e:
    print(f"API Exception occurred: {e}")


Success! Message ID: <202607050738.59965351328@smtp-relay.mailin.fr>


In [ ]:
from langchain_core.tools import tool
import sib_api_v3_sdk
from sib_api_v3_sdk.rest import ApiException

@tool
def sendMail(
    sender: str,
    receiver: str,
    subject: str,
    body: str
) -> str:
    """
    Send an email using Brevo.
    """

    configuration = sib_api_v3_sdk.Configuration()
    configuration.api_key["api-key"] = BREVO_KEY

    api_instance = sib_api_v3_sdk.TransactionalEmailsApi(
        sib_api_v3_sdk.ApiClient(configuration)
    )

    send_smtp_email = sib_api_v3_sdk.SendSmtpEmail(
        sender={ "name": "LOOKOUT",
        "email": "lokeshvijayraina@gmail.com"},
        to=[{"email": receiver}],
        subject=sub,
        html_content=body,
    )

    try:
        api_response = api_instance.send_transac_email(send_smtp_email)
        print("Success! Message ID:", api_response.message_id)

        return f"Mail sent successfully to {receiver}"

    except ApiException as e:
        print(" Error:", e)

        return f"Failed to send email: {e}"

In [35]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
agent = create_agent(
    model=llm_groq,
    tools=[sendMail],
    system_prompt=
    """ 
  You are LOOKOUT, an AI email assistant that drafts and sends professional emails on behalf of the user.

Guidelines:
- Always write in a clear, professional, and friendly tone.
- Keep emails concise: 3-5 short paragraphs maximum, no filler.
- Use the sender name provided to sign off the email appropriately.
- Include a relevant subject line if one is not explicitly given.
- Format the email body as clean HTML (use <p> tags for paragraphs, avoid unnecessary styling).
- Do not invent facts, names, or details that were not provided to you — ask for clarification only if critical information is missing.
- Once the email content is ready, call the sendMail tool with the correct sender, receiver, subject, and body.
- After sending, confirm success or failure back to the user in one short sentence.
 """
)

response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="""
Task: Draft and send an email.

Sender email: lokeshvijayraina@gmail.com
Sender name: Tyler Durden
Receiver email: itslokeshx@gmail.com
Subject: Welcome

Context/instructions: Write a warm welcome email to a new user of the LOOKOUT platform. Mention that we're excited to have them onboard and that they can reach out with any questions. Sign off as Tyler Durden.

Once drafted, send it using the sendMail tool."""
            )
        ]
    }
)

print(response["messages"][-1].content)

Success! Message ID: <202607050802.21926291598@smtp-relay.mailin.fr>
(Note: Since we've already used the sendMail function to send the email, there's no additional code needed. The task is complete.)
